### Import Libraries

In this step, we import the libraries required for loading the processed dataset, splitting the data into training, validation, and test sets, performing TF-IDF vectorization, and saving the trained vectorizer for later use.

In [1]:
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

### Load Processed Dataset

The cleaned dataset generated during preprocessing is loaded for feature engineering.

In [2]:
data_path = Path("../data/processed/processed_news.csv")

df = pd.read_csv(data_path)

print(df.shape)

df.head()

(38653, 2)


,content,label
0,ben stein call th circuit court committed ‘cou...,1
1,trump drop steve bannon national security coun...,0
2,puerto rico expects u lift jones act shipping ...,0
3,oops trump accidentally confirmed leaked israe...,1
4,donald trump head scotland reopen golf resort ...,0


### Separate Features and Target

The `content` column is used as the input feature, while the `label` column is used as the target variable.

In [3]:
X = df["content"]
y = df["label"]

print(X.shape)
print(y.shape)

(38653,)
(38653,)


In [4]:
print(df.isnull().sum())

content    0
label      0
dtype: int64


In [5]:
# Remove rows with missing content
df = df.dropna(subset=["content"]).reset_index(drop=True)

print(df.isnull().sum())
print(df.shape)

content    0
label      0
dtype: int64
(38653, 2)


In [6]:
X = df["content"]
y = df["label"]

### Split the Dataset

The dataset is divided into training (70%), validation (10%), and testing (20%) sets using stratified sampling to preserve the class distribution across all splits.

In [7]:
# First split: 80% train+validation, 20% test

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split:
# From the remaining 80%, allocate 12.5% to validation.
# 12.5% of 80% = 10% of the original dataset.

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.125,
    random_state=42,
    stratify=y_train_val
)

In [8]:
print("Training Set :", X_train.shape[0])
print("Validation Set :", X_val.shape[0])
print("Test Set :", X_test.shape[0])

Training Set : 27056
Validation Set : 3866
Test Set : 7731


### TF-IDF Vectorization


In [9]:
# Create TF-IDF vectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Fit only on training data
X_train_tfidf = vectorizer.fit_transform(X_train)

# Transform validation and test data
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

In [10]:
print(X_train.isnull().sum())
print(X_val.isnull().sum())
print(X_test.isnull().sum())

0
0
0


### Inspect Feature Matrix

After applying TF-IDF vectorization, each news article is represented as a numerical feature vector. We inspect the dimensions of the transformed datasets to verify that the feature extraction process has been completed successfully.

In [11]:
print("Training set shape:", X_train_tfidf.shape)
print("Validation set shape:", X_val_tfidf.shape)
print("Test set shape:", X_test_tfidf.shape)

Training set shape: (27056, 20000)
Validation set shape: (3866, 20000)
Test set shape: (7731, 20000)


### Vocabulary Size

The TF-IDF vectorizer builds a vocabulary from the training data. This step shows the number of unique terms selected after applying the specified filtering criteria.

In [12]:
print("Vocabulary size:", len(vectorizer.vocabulary_))

Vocabulary size: 20000


### Save TF-IDF Vectorizer

The trained TF-IDF vectorizer is saved so that the same feature transformation can be used during model evaluation and prediction on unseen news articles.

In [13]:
artifact_path = Path("../artifacts")
artifact_path.mkdir(parents=True, exist_ok=True)

joblib.dump(vectorizer, artifact_path / "tfidf_vectorizer.pkl")

print("TF-IDF vectorizer saved successfully.")

TF-IDF vectorizer saved successfully.


In [14]:
print((artifact_path / "tfidf_vectorizer.pkl").exists())

True
